In [1]:
# Make sure this GitHub repo contains code/evaluate_cross_dataset.py before running on Kaggle.
# If you upload the edited repo as a Kaggle Dataset instead, replace this cell with a cp -r command.
REPO_URL = "https://github.com/DinhLiu/Generalizable-FER.git"
!git clone {REPO_URL} /kaggle/working/Generalizable-FER

Cloning into '/kaggle/working/Generalizable-FER'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 178 (delta 14), reused 37 (delta 7), pack-reused 129 (from 2)
Receiving objects: 100% (178/178), 168.75 MiB | 38.33 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [2]:
%cd /kaggle/working/Generalizable-FER

/kaggle/working/Generalizable-FER


In [3]:
%%capture
!pip install -r requirements.txt

## Kaggle input structure

This notebook expects the Kaggle inputs from `main.pdf`:

- FERPlus: `test/train/validation` with class folders, including `suprise` and `contempt`.
- AffectNet: `archive (3)/Test` and `archive (3)/Train` with class folders.
- SFEW2.0: `Train` and `Val` with class folders; `Test/Test_Aligned_Faces` is unlabeled, so evaluation uses `Val`.
- MMAFEDB: `MMAFEDB/test`, `MMAFEDB/train`, and `MMAFEDB/valid` with class folders.

`contempt` is skipped because the CAFE checkpoint trained on RAF-DB has 7 outputs and no contempt class.

In [4]:
from pathlib import Path
import os
import shutil

WORK_DATA = Path("/kaggle/working/data")
WORK_DATA.mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/outputs").mkdir(parents=True, exist_ok=True)

LINKS = {
    "raf-basic": [
        "/kaggle/input/raf-db-dataset",
        "/kaggle/input/datasets/shuvoalok/raf-db-dataset",
        "/kaggle/input/raf-db/raf-basic",
    ],
    "ckplus48": [
        "/kaggle/input/ckdataset",
        "/kaggle/input/ckplus48",
        "/kaggle/input/datasets/davilsena/ckdataset",
    ],
    "ferplus": [
        "/kaggle/input/ferplus",
        "/kaggle/input/datasets/arnabkumarroy02/ferplus",
    ],
    "affectnet": [
        "/kaggle/input/affectnet",
        "/kaggle/input/datasets/mstjebashazida/affectnet",
    ],
    "sfew": [
        "/kaggle/input/datasetsfew",
        "/kaggle/input/dataset-sfew",
        "/kaggle/input/datasets/vlntnstarodub/datasetsfew",
    ],
    "mma": [
        "/kaggle/input/mma-facial-expression",
        "/kaggle/input/datasets/mahmoudima/mma-facial-expression",
    ],
    "resnet18_msceleb.pth": [
        "/kaggle/input/resnet18-msceleb/resnet18_msceleb.pth",
        "/kaggle/input/datasets/vnnhhiu/resnet18-msceleb/resnet18_msceleb.pth",
    ],
}
OPTIONAL = {"ckplus48"}


def first_existing(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    return None


def replace_link(dst, src):
    dst = Path(dst)
    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    os.symlink(src, dst, target_is_directory=src.is_dir())

missing = []
for name, candidates in LINKS.items():
    src = first_existing(candidates)
    if src is None:
        if name not in OPTIONAL:
            missing.append((name, candidates))
        print(f"SKIP {name}: not found")
        continue
    dst = Path("/kaggle/working") / name if name.endswith(".pth") else WORK_DATA / name
    replace_link(dst, src)
    print(f"{name}: {src} -> {dst}")

if missing:
    msg = "\n".join(f"{name}: {candidates}" for name, candidates in missing)
    raise FileNotFoundError(f"Missing required Kaggle inputs:\n{msg}")

raf-basic: /kaggle/input/datasets/shuvoalok/raf-db-dataset -> /kaggle/working/data/raf-basic
ckplus48: /kaggle/input/datasets/davilsena/ckdataset -> /kaggle/working/data/ckplus48
ferplus: /kaggle/input/datasets/arnabkumarroy02/ferplus -> /kaggle/working/data/ferplus
affectnet: /kaggle/input/datasets/mstjebashazida/affectnet -> /kaggle/working/data/affectnet
sfew: /kaggle/input/datasets/vlntnstarodub/datasetsfew -> /kaggle/working/data/sfew
mma: /kaggle/input/datasets/mahmoudima/mma-facial-expression -> /kaggle/working/data/mma
resnet18_msceleb.pth: /kaggle/input/datasets/vnnhhiu/resnet18-msceleb/resnet18_msceleb.pth -> /kaggle/working/resnet18_msceleb.pth


In [5]:
from pathlib import Path

for root in [
    Path("/kaggle/working/data/ferplus"),
    Path("/kaggle/working/data/affectnet"),
    Path("/kaggle/working/data/sfew"),
    Path("/kaggle/working/data/mma"),
]:
    print(f"\n== {root} ==")
    for path in sorted(p for p in root.rglob("*") if p.is_dir()):
        rel = path.relative_to(root)
        if len(rel.parts) <= 3:
            print(rel)


== /kaggle/working/data/ferplus ==
test
test/angry
test/contempt
test/disgust
test/fear
test/happy
test/neutral
test/sad
test/suprise
train
train/angry
train/contempt
train/disgust
train/fear
train/happy
train/neutral
train/sad
train/suprise
validation
validation/angry
validation/contempt
validation/disgust
validation/fear
validation/happy
validation/neutral
validation/sad
validation/suprise

== /kaggle/working/data/affectnet ==
archive (3)
archive (3)/Test
archive (3)/Test/Anger
archive (3)/Test/Contempt
archive (3)/Test/disgust
archive (3)/Test/fear
archive (3)/Test/happy
archive (3)/Test/neutral
archive (3)/Test/sad
archive (3)/Test/surprise
archive (3)/Train
archive (3)/Train/anger
archive (3)/Train/contempt
archive (3)/Train/disgust
archive (3)/Train/fear
archive (3)/Train/happy
archive (3)/Train/neutral
archive (3)/Train/sad
archive (3)/Train/surprise

== /kaggle/working/data/sfew ==
Test
Test/Test_Aligned_Faces
Train
Train/Angry
Train/Disgust
Train/Fear
Train/Happy
Train/Neutra

In [6]:
%cd /kaggle/working/Generalizable-FER

!python scripts/create_raf_compatible.py \
  --src /kaggle/working/data/raf-basic \
  --dst /kaggle/working/data/raf-basic-compatible

/kaggle/working/Generalizable-FER
Created /kaggle/working/data/raf-basic-compatible
Train: 12271 kept_1_to_7 {1: 1290, 2: 281, 3: 717, 4: 4772, 5: 1982, 6: 705, 7: 2524}
Test: 3068 kept_1_to_7 {1: 329, 2: 74, 3: 160, 4: 1185, 5: 478, 6: 162, 7: 680}


In [7]:
# %cd /kaggle/working/Generalizable-FER/code
# Smoke test:
# !python ours_CAFE.py \
#   --raf_path ../../data/raf-basic-compatible \
#   --label_path list_patition_label.txt \
#   --batch_size 32 \
#   --workers 2 \
#   --gpu 0 \
#   --epochs 1

In [8]:
%cd /kaggle/working/Generalizable-FER/code
!python ours_CAFE.py \
  --raf_path ../../data/raf-basic-compatible \
  --label_path list_patition_label.txt \
  --batch_size 32 \
  --workers 2 \
  --gpu 0 \
  --epochs 60

/kaggle/working/Generalizable-FER/code
100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 153MiB/s]
weight : torch.Size([7, 512])
bias : torch.Size([7])
epoch:  1 acc:  tensor(0.6910, device='cuda:0')
epoch:  2 acc:  tensor(0.7676, device='cuda:0')
epoch:  3 acc:  tensor(0.8067, device='cuda:0')
epoch:  4 acc:  tensor(0.8246, device='cuda:0')
epoch:  5 acc:  tensor(0.8462, device='cuda:0')
epoch:  6 acc:  tensor(0.8517, device='cuda:0')
epoch:  7 acc:  tensor(0.8618, device='cuda:0')
epoch:  8 acc:  tensor(0.8716, device='cuda:0')
epoch:  9 acc:  tensor(0.8745, device='cuda:0')
epoch:  10 acc:  tensor(0.8791, device='cuda:0')
epoch:  11 acc:  tensor(0.8820, device='cuda:0')
epoch:  12 acc:  tensor(0.8843, device='cuda:0')
epoch:  13 acc:  tensor(0.8768, device='cuda:0')
epoch:  14 acc:  tensor(0.8840, device='cuda:0')
epoch:  15 acc:  tensor(0.8833, device='cuda:0')
epoch:  16 acc:  tensor(0.8833, device='cuda:0')
epoch:  17 acc:  tensor(0.8820, device='cuda:0')
epo

In [9]:
!mkdir -p /kaggle/working/outputs/rafdb_cafe_seed3407
!cp ours_best.pth ours_final.pth results.txt /kaggle/working/outputs/rafdb_cafe_seed3407/

In [10]:
!zip -r /kaggle/working/outputs/rafdb_cafe_seed3407.zip /kaggle/working/outputs/rafdb_cafe_seed3407

  adding: kaggle/working/outputs/rafdb_cafe_seed3407/ (stored 0%)
  adding: kaggle/working/outputs/rafdb_cafe_seed3407/results.txt (deflated 86%)
  adding: kaggle/working/outputs/rafdb_cafe_seed3407/ours_final.pth (deflated 7%)
  adding: kaggle/working/outputs/rafdb_cafe_seed3407/ours_best.pth (deflated 7%)


In [11]:
import subprocess
from pathlib import Path

CODE_DIR = Path("/kaggle/working/Generalizable-FER/code")
CHECKPOINT = "/kaggle/working/outputs/rafdb_cafe_seed3407/ours_best.pth"
ck_path = Path("/kaggle/working/data/ckplus48")

if ck_path.exists():
    subprocess.run([
        "python", "evaluate_ckplus48.py",
        "--ck_path", str(ck_path),
        "--checkpoint", CHECKPOINT,
        "--output_dir", "/kaggle/working/outputs/ckplus48_eval",
        "--batch_size", "64",
        "--workers", "2",
        "--gpu", "0",
    ], cwd=CODE_DIR, check=True)
    subprocess.run([
        "python", "evaluate_ckplus48.py",
        "--ck_path", str(ck_path),
        "--checkpoint", CHECKPOINT,
        "--output_dir", "/kaggle/working/outputs/ckplus48_peak_eval",
        "--batch_size", "64",
        "--workers", "2",
        "--gpu", "0",
        "--exclude_neutral",
    ], cwd=CODE_DIR, check=True)
else:
    print("CK+48 input is not linked; skipping CK+48 evaluation.")


--- Dataset Class Distribution ---
  anger: 45
  disgust: 59
  fear: 25
  happy: 69
  neutral: 593
  sadness: 28
  surprise: 83
Total samples: 902

weight : torch.Size([7, 512])
bias : torch.Size([7])


/kaggle/working/Generalizable-FER/code/evaluate_ckplus48.py:105: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(image, mode="L").convert("RGB")
/kaggle/working/Generalizable-FER/code/evaluate_ckplus48.py:105: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(image, mode="L").convert("RGB")


{
  "dataset_mode": "CK+48-7cls-with-neutral",
  "num_samples": 902,
  "labels": [
    "surprise",
    "fear",
    "disgust",
    "happy",
    "sadness",
    "anger",
    "neutral"
  ],
  "overall_accuracy": 0.8248337028824834,
  "mean_accuracy": 0.5790145351453543,
  "macro_f1": 0.5372975734972046,
  "per_class_accuracy": {
    "surprise": 0.9518072289156626,
    "fear": 0.0,
    "disgust": 0.559322033898305,
    "happy": 1.0,
    "sadness": 0.5357142857142857,
    "anger": 0.08888888888888889,
    "neutral": 0.9173693086003373
  },
  "class_counts": {
    "neutral": 593,
    "happy": 69,
    "anger": 45,
    "disgust": 59,
    "sadness": 28,
    "fear": 25,
    "surprise": 83
  }
}
Wrote outputs to /kaggle/working/outputs/ckplus48_eval

--- Dataset Class Distribution ---
  anger: 45
  disgust: 59
  fear: 25
  happy: 69
  sadness: 28
  surprise: 83
Total samples: 309

weight : torch.Size([7, 512])
bias : torch.Size([7])


/kaggle/working/Generalizable-FER/code/evaluate_ckplus48.py:105: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(image, mode="L").convert("RGB")
/kaggle/working/Generalizable-FER/code/evaluate_ckplus48.py:105: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(image, mode="L").convert("RGB")


{
  "dataset_mode": "CK+48-6cls-peak-only",
  "num_samples": 309,
  "labels": [
    "surprise",
    "fear",
    "disgust",
    "happy",
    "sadness",
    "anger"
  ],
  "overall_accuracy": 0.6472491909385113,
  "mean_accuracy": 0.522622072902857,
  "macro_f1": 0.5143157592391283,
  "per_class_accuracy": {
    "surprise": 0.9518072289156626,
    "fear": 0.0,
    "disgust": 0.559322033898305,
    "happy": 1.0,
    "sadness": 0.5357142857142857,
    "anger": 0.08888888888888889
  },
  "class_counts": {
    "happy": 69,
    "anger": 45,
    "disgust": 59,
    "sadness": 28,
    "fear": 25,
    "surprise": 83
  }
}
Wrote outputs to /kaggle/working/outputs/ckplus48_peak_eval


In [12]:
import subprocess
from pathlib import Path

CODE_DIR = Path("/kaggle/working/Generalizable-FER/code")
CHECKPOINT = "/kaggle/working/outputs/rafdb_cafe_seed3407/ours_best.pth"

PAPER_EVALS = [
    ("ferplus", "/kaggle/working/data/ferplus", None, "/kaggle/working/outputs/ferplus_eval"),
    ("affectnet", "/kaggle/working/data/affectnet", None, "/kaggle/working/outputs/affectnet_eval"),
    ("sfew", "/kaggle/working/data/sfew", "Val", "/kaggle/working/outputs/sfew_val_eval"),
    ("mma", "/kaggle/working/data/mma", None, "/kaggle/working/outputs/mma_eval"),
]

for dataset, data_path, split, output_dir in PAPER_EVALS:
    cmd = [
        "python", "evaluate_cross_dataset.py",
        "--dataset", dataset,
        "--data_path", data_path,
        "--checkpoint", CHECKPOINT,
        "--output_dir", output_dir,
        "--batch_size", "64",
        "--workers", "2",
        "--gpu", "0",
    ]
    if split:
        cmd.extend(["--split", split])
    print("\nRunning", dataset, "...")
    subprocess.run(cmd, cwd=CODE_DIR, check=True)


Running ferplus ...
weight : torch.Size([7, 512])
bias : torch.Size([7])
{
  "dataset": "ferplus",
  "dataset_mode": "ferplus-test-7cls-no-contempt",
  "num_samples": 3543,
  "labels": [
    "surprise",
    "fear",
    "disgust",
    "happy",
    "sadness",
    "anger",
    "neutral"
  ],
  "overall_accuracy": 0.668077900084674,
  "mean_accuracy": 0.5624254736745612,
  "macro_f1": 0.5305761505246683,
  "per_class_accuracy": {
    "surprise": 0.6822222222222222,
    "fear": 0.3163265306122449,
    "disgust": 0.2857142857142857,
    "happy": 0.9451022604951561,
    "sadness": 0.6792873051224945,
    "anger": 0.4937888198757764,
    "neutral": 0.5345368916797488
  },
  "class_counts": {
    "anger": 322,
    "disgust": 21,
    "fear": 98,
    "happy": 929,
    "neutral": 1274,
    "sadness": 449,
    "surprise": 450
  },
  "skipped_classes": {
    "contempt": 30
  },
  "confusion_matrix_labels": [
    "surprise",
    "fear",
    "disgust",
    "happy",
    "sadness",
    "anger",
    "ne

In [13]:
import json
from pathlib import Path
import pandas as pd

rows = []
for metrics_path in sorted(Path("/kaggle/working/outputs").glob("*_eval/metrics.json")):
    metrics = json.loads(metrics_path.read_text())
    rows.append({
        "dataset": metrics.get("dataset", metrics.get("dataset_mode", metrics_path.parent.name)),
        "dataset_mode": metrics.get("dataset_mode"),
        "num_samples": metrics.get("num_samples"),
        "overall_accuracy": metrics.get("overall_accuracy"),
        "mean_accuracy": metrics.get("mean_accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "skipped_classes": metrics.get("skipped_classes", {}),
        "output_dir": str(metrics_path.parent),
    })

summary = pd.DataFrame(rows).sort_values("dataset_mode")
summary.to_csv("/kaggle/working/outputs/cross_dataset_summary.csv", index=False)
display(summary)

,dataset,dataset_mode,num_samples,overall_accuracy,mean_accuracy,macro_f1,skipped_classes,output_dir
2,CK+48-6cls-peak-only,CK+48-6cls-peak-only,309,0.647249,0.522622,0.514316,{},/kaggle/working/outputs/ckplus48_peak_eval
1,CK+48-7cls-with-neutral,CK+48-7cls-with-neutral,902,0.824834,0.579015,0.537298,{},/kaggle/working/outputs/ckplus48_eval
0,affectnet,affectnet-Test-7cls-no-contempt,13206,0.520142,0.469332,0.439706,{'Contempt': 1312},/kaggle/working/outputs/affectnet_eval
3,ferplus,ferplus-test-7cls-no-contempt,3543,0.668078,0.562425,0.530576,{'contempt': 30},/kaggle/working/outputs/ferplus_eval
4,mma,mma-test-7cls-no-contempt,17356,0.567181,0.429504,0.431677,{},/kaggle/working/outputs/mma_eval
5,sfew,sfew-Val-7cls-no-contempt,431,0.468677,0.387789,0.339830,{},/kaggle/working/outputs/sfew_val_eval


In [14]:
!cd /kaggle/working && zip -r outputs/cross_dataset_evaluations.zip outputs/*_eval outputs/cross_dataset_summary.csv

  adding: outputs/affectnet_eval/ (stored 0%)
  adding: outputs/affectnet_eval/metrics.json (deflated 65%)
  adding: outputs/affectnet_eval/predictions.csv (deflated 87%)
  adding: outputs/affectnet_eval/confusion_matrix.png (deflated 11%)
  adding: outputs/ckplus48_eval/ (stored 0%)
  adding: outputs/ckplus48_eval/metrics.json (deflated 55%)
  adding: outputs/ckplus48_eval/class_distribution.png (deflated 26%)
  adding: outputs/ckplus48_eval/predictions.csv (deflated 75%)
  adding: outputs/ckplus48_eval/confusion_matrix.png (deflated 16%)
  adding: outputs/ckplus48_peak_eval/ (stored 0%)
  adding: outputs/ckplus48_peak_eval/metrics.json (deflated 52%)
  adding: outputs/ckplus48_peak_eval/class_distribution.png (deflated 27%)
  adding: outputs/ckplus48_peak_eval/predictions.csv (deflated 73%)
  adding: outputs/ckplus48_peak_eval/confusion_matrix.png (deflated 16%)
  adding: outputs/ferplus_eval/ (stored 0%)
  adding: outputs/ferplus_eval/metrics.json (deflated 65%)
  adding: outputs/fe